In [ ]:
import numpy
import pandas

import matplotlib.pyplot as plt
plt.style.use('mystyle.mplstyle')

import sbruceana

In [ ]:
PATH_TO_SBRUCE = "/Users/triozzi/Analysis/nuedis/checks/data/ncpi0/"

In [ ]:
MASS_PI0 = 134.9766

#### Selection

In [ ]:
FILE_CV = "cv/CNAF_CV_NCpi0_NuMI_NoSysts.root"
FILE_OFFBEAM = ""
FILE_DATA = "cv/CNAF_Data_NCpi0_NuMI_NoSysts.root"

In [ ]:
df = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_CV}",
  "events/selectedNu"  
)
pot = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{FILE_CV}")

df_cos = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_CV}",
  "events/selectedCos"  
)
pot_cos = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{FILE_CV}")


df_data = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_DATA}",
  "events/selectedData"  
)
pot_data = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{FILE_DATA}")

In [ ]:
df_cos['cosmic'] = 1
df = pandas.concat(
  (df, df_cos)
)

In [ ]:
# fig, ax = plt.subplots(figsize=(4.25, 3.25), layout='constrained')
fig, ax = plt.subplots(figsize=(4.25, 3.75), layout='constrained')


var = "pi0mass"

width = 30; bins = numpy.arange(0, 600+width, width)

ax = sbruceana.plotting.plot_by_category(ax, df, sbruceana.plotting.NCPI0_CATEGORIES, bins, var, True, True)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var)

ax.axvline(MASS_PI0, zorder=3, lw=0.75, c='black', label='134.9766 MeV')

# gfx
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = '$M_{\\pi^0}$ [MeV]',
  ylabel = f'slices [a.n.]\n/ {width} MeV',
  xlim   = (bins[0], bins[-1]),
)
# leg = ax.legend(fontsize=9.5, title='NuMI CV'); leg.get_title().set_fontsize(10)
leg = ax.legend(fontsize=8.5)

fig.suptitle("All uncorrected", fontsize=20)

plt.show()
fig.savefig(f"plots/ncpi0/uncorrected_{var}.pdf", dpi=300)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def chi2_histograms(df, df_data, var, bins, factor=1.0):
    """
    Compute chi-squared between MC and scaled-data histograms using Poisson errors.
    The factor is applied to the x-values of the data: df_data[var] * factor.
    """
    mc_counts,   _ = np.histogram(df[var],               bins=bins)
    data_counts, _ = np.histogram(df_data[var] * factor, bins=bins)

    mc_norm   = mc_counts   / mc_counts.sum()
    data_norm = data_counts / data_counts.sum()

    mask = mc_norm > 0

    raw_clipped = np.clip(data_counts[mask], 1, None)
    sigma2 = data_norm[mask] / raw_clipped

    chi2_val = np.sum((mc_norm[mask] - data_norm[mask]) ** 2 / sigma2)
    ndof     = mask.sum()

    return chi2_val, ndof


def sweep_chi2(df, df_data, var, bins, factors=None, plot=True):
    """
    Sweep a multiplicative factor on df_data[var], minimising chi-squared.
    Returns the best factor and the full chi2 array.
    """
    if factors is None:
        factors = np.linspace(0.5, 2.0, 200)

    chi2_vals = np.array([chi2_histograms(df, df_data, var, bins, f)[0]
                          for f in factors])

    best_factor = factors[np.argmin(chi2_vals)]
    _, ndof = chi2_histograms(df, df_data, var, bins, best_factor)

    if plot:
        fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), layout='constrained')

        # --- left: chi2 vs factor ---
        ax = axes[0]
        ax.plot(factors, chi2_vals, lw=1.5)
        ax.axvline(best_factor, color='C1', ls='--',
                   label=f'best = {best_factor:.3f}')
        ax.axhline(ndof, color='C2', ls=':',
                   label=f'ndof = {ndof}')
        ax.set_xlabel('Scale factor on data x-values')
        ax.set_ylabel('$\\chi^2$')
        ax.set_title(f'$\\chi^2$ sweep — {var}')
        ax.legend()

        # --- right: overlay at best factor ---
        ax = axes[1]
        ax.hist(df[var],                        bins=bins, density=True,
                alpha=0.6, label='MC')
        ax.hist(df_data[var] * best_factor,     bins=bins, density=True,
                histtype='step', linewidth=2,
                label=f'Data × {best_factor:.3f}')
        ax.set_xlabel(var)
        ax.set_ylabel('Density')
        ax.set_title(f'Best-fit overlay — {var}')
        ax.legend()

        plt.show()

    return best_factor, chi2_vals

In [ ]:
# --- usage ---
var   = "pi0mass"
width = 30; bins = numpy.arange(0, 600+width, width)

best_factor, chi2_curve = sweep_chi2(df, df_data, var, bins)

c2, ndof = chi2_histograms(df, df_data, var, bins, best_factor)
print(f"Best factor: {best_factor:.4f}")
print(f"chi2 = {c2:.2f},  ndof = {ndof},  chi2/ndof = {c2/ndof:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.75), layout='constrained')

var = "pi0mass"

width = 30; bins = numpy.arange(0, 600+width, width)

ax = sbruceana.plotting.plot_by_category(ax, df, sbruceana.plotting.NCPI0_CATEGORIES, bins, var, True, True)
ax = sbruceana.plotting.plot_data(ax, df_data*0.879, bins, var)

ax.axvline(MASS_PI0, zorder=3, lw=0.75, c='black', label='134.9766 MeV')

# gfx
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = '$M_{\\pi^0}$ [MeV]',
  ylabel = f'slices [a.n.]\n/ {width} MeV',
  xlim   = (bins[0], bins[-1]),
)
# leg = ax.legend(fontsize=9.5, title='NuMI CV'); leg.get_title().set_fontsize(10)
leg = ax.legend(fontsize=8.5)

fig.suptitle("Data * 0.879", fontsize=20)

plt.show()
fig.savefig(f"plots/ncpi0/datacorrected_{var}.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.75), layout='constrained')

var = "pi0mass"

width = 30; bins = numpy.arange(0, 600+width, width)

ax = sbruceana.plotting.plot_by_category(ax, df, sbruceana.plotting.NCPI0_CATEGORIES, bins, var, 1.13, True, True)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var)

ax.axvline(MASS_PI0, zorder=3, lw=0.75, c='black', label='134.9766 MeV')

# gfx
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = '$M_{\\pi^0}$ [MeV]',
  ylabel = f'slices [a.n.]\n/ {width} MeV',
  xlim   = (bins[0], bins[-1]),
)
# leg = ax.legend(fontsize=9.5, title='NuMI CV'); leg.get_title().set_fontsize(10)
leg = ax.legend(fontsize=8.)

fig.suptitle("MC * 1.121", fontsize=20)

plt.show()
fig.savefig(f"plots/preselection/preselection_allcorrected_{var}.pdf", dpi=300)

In [ ]:
df.columns

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.25), layout='constrained')

var = "pi0open"

width = 0.1; bins = numpy.arange(-1, 1+width, width)

ax = sbruceana.plotting.plot_by_category(ax, df, sbruceana.plotting.NCPI0_CATEGORIES, bins, var, True, True)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var)

ax.axvline(MASS_PI0, zorder=3, lw=0.75, c='black', label='134.9766 MeV')

# gfx
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'cos($\\theta_{\\gamma\\gamma}$)',
  ylabel = f'slices [a.n.]\n/ {width}',
  xlim   = (bins[0], bins[-1]),
)
# leg = ax.legend(fontsize=9.5, title='NuMI CV'); leg.get_title().set_fontsize(10)
leg = ax.legend(fontsize=8., ncol=3)

plt.show()
fig.savefig(f"plots/ncpi0/{var}.pdf", dpi=300)